In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# =============================================================================

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install -q -U transformers accelerate huggingface_hub supervision opencv-python-headless

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # uses HF_TOKEN from Colab secrets, or paste your token (needs access to facebook/sam3)

print("Environment ready.")


In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time

import numpy as np
import pandas as pd
import torch

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor

# UAV orthomosaics are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)

In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# PIPELINE - TWO SAM3 ROUNDS PER IMAGE (no tiling)
#   ROUND 1: the WHOLE UAV image (downscaled to MAX_DIM) is sent to SAM3 with the
#            K_EXEMPLARS = 3 size-based GT boxes (S, M, L) as positive prompts.
#   SELF-PROMPTS: the round-1 detections are NMS-merged and the ones that fall on
#            a prompt plant are dropped (IoU >= SELF_PROMPT_EXCLUDE_IOU with any
#            S/M/L box). From the remaining boxes:
#              POSITIVE self-prompts = the N_SELF_POSITIVES = 2 HIGHEST-confidence
#                  boxes (chosen first, so they can never also become negatives),
#              NEGATIVE self-prompts = boxes that are not reliable enough to call
#                  Rumex, i.e. confidence < UNRELIABLE_MAX_SCORE = 0.50; they are
#                  ranked by confidence and the N_SELF_NEGATIVES = 2 LOWEST ones
#                  are taken.
#            All of these are SAM3's own predictions, not GT.
#   ROUND 2: the same image is sent to SAM3 again in ONE forward pass with
#            3 + 2 positive boxes (label 1) and 2 negative boxes (label 0).
#   The FINAL predictions are the round-2 output (round 1 is NOT merged in); they
#   are evaluated against the original human GT annotations.
#            Whatever exists is used: 2 positives + 1 negative, negatives only, and
#            so on. Only when NEITHER a positive NOR a negative self-prompt exists
#            would round 2 repeat round 1 exactly, so it is skipped and the round-1
#            output is the final result (second_run=False).
#
# SIZE-BASED EXEMPLARS (CELL 10) - chosen from ALL GT boxes of the image by AREA
# (width x height), every GT box counts (including boxes cut by the image border):
#   S = smallest, L = largest, M = the remaining box whose area is closest to the
#   median area of all GT boxes of that image. Deterministic -> ONE run per image.
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "pos_neg_pseudo_prompts_no_tiling"
K_EXEMPLARS      = 3          # S + M + L (GT boxes, round 1 and round 2)
N_SELF_POSITIVES = 2          # highest-confidence round-1 predictions -> POSITIVE in round 2
N_SELF_NEGATIVES = 2          # lowest-confidence UNRELIABLE round-1 predictions -> NEGATIVE
N_EXEMPLARS      = K_EXEMPLARS
SIZE_MEASURE    = "area"      # box size = width x height (full-resolution pixels)
USE_TILING      = False       # whole image, no tiles
PROMPT_TYPE      = f"size_SML+{N_SELF_POSITIVES}pos_self+{N_SELF_NEGATIVES}neg_self"

# --------------------- self-prompt selection (round 1 -> 2) -------------------
SELF_PROMPT_EXCLUDE_IOU = 0.50   # a round-1 prediction whose IoU with ANY S/M/L
                                 # prompt box is >= this value is NOT eligible as a
                                 # self-prompt (it is just the prompt plant again)
UNRELIABLE_MAX_SCORE    = 0.50   # "not reliable enough to confidently call Rumex":
                                 # a round-1 prediction with confidence BELOW this
                                 # value may be used as a NEGATIVE prompt in round 2.
                                 # The pool is therefore [CONFIDENCE_THRESHOLD, 0.50).
                                 # The positive self-prompts are picked FIRST and are
                                 # removed from this pool, so no box is ever both.

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_NAME}"

# --------------------------- whole-image resize -------------------------------
MAX_DIM = 1024                 # resize cap for SAM3 input (no tiling)

# --------------------------- SAM3 inference ----------------------------------
# The confidence threshold is FIXED from the start. It is used both
#   (a) inside SAM3 post-processing (BOTH rounds), and
#   (b) as the operating point for precision / recall / F1 / IoU1 / IoU2.
# Because (a) == (b), AP and the operating-point metrics use the same predictions.
CONFIDENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40          # SAM3 mask binarisation (kept as in the E02_1 code)
KEEP_MASKS = False              # masks are NEVER stored (RAM / GPU / disk / CSV / NPZ)

# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
PROMPT_IGNORE_IOU  = 0.50     # held_out mode: an unmatched prediction whose best IoU
                              # with a PROMPT GT box is >= this value is IGNORED
                              # (neither TP nor FP). Same value as EVAL_IOU_THRESHOLD
                              # so a single IoU threshold governs the whole protocol.

# ------------------------------- NMS -----------------------------------------
# Fixed from the start (no confidence x NMS sweep, no "best" selection).
# Used twice: on the round-1 detections to pick the self-prompts (CELL 13) and on
# the final detections during evaluation.
NMS_IOU_THRESHOLD = 0.40

EVALUATION_MODES = ["all_gt", "held_out"]
# all_gt   : every GT box of the image is evaluated - this is the main number
#            ("final predictions vs ALL original human GT annotations").
# held_out : the 3 GT instances used as S/M/L prompts are IGNORED, and so are the
#            predictions that fall on them. The self-prompts are NOT GT boxes, so
#            they never change the GT set.

# --------------------------- qualitative plot ---------------------------------
PLOT_EVALUATION_MODE = "all_gt"   # per-image AP50 of this mode selects the image
PLOT_MIN_GT_BOXES    = 7          # the plotted image must have at least this many GT boxes
PLOT_MAX_DISPLAY_DIM = 2048       # display-only downscale (the full image is 8192 px wide)
PLOT_SHOW_SCORES     = True       # write the confidence next to each predicted box

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  results folder : {RESULTS_ROOT}")
print(f"  round 1        : {K_EXEMPLARS} GT boxes by {SIZE_MEASURE} (smallest / medium / largest)")
print(f"  round 2        : same 3 GT boxes + up to {N_SELF_POSITIVES} POSITIVE self-prompts "
      f"(highest confidence) + up to {N_SELF_NEGATIVES} NEGATIVE self-prompts "
      f"(lowest confidence, below {UNRELIABLE_MAX_SCORE})")
print(f"  evaluated      : round-2 output vs the original GT ({EVALUATION_MODES}, "
      f"IoU >= {EVAL_IOU_THRESHOLD})")
print(f"  tiling         : {USE_TILING} (whole image resized to MAX_DIM={MAX_DIM})")
print(f"  confidence     : {CONFIDENCE_THRESHOLD} (fixed, both rounds)")
print(f"  mask threshold : {MASK_THRESHOLD}")
print(f"  NMS IoU        : {NMS_IOU_THRESHOLD} (fixed)")
print(f"  masks stored   : {KEEP_MASKS}")

In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results_new1/<EXPERIMENT_NAME>/
#   raw_detections/      pre-NMS detections (NPZ, one file per image)
#   metrics/             per-image table, experiment summary, pooled dataset AP,
#                        size-group recall
#   confusion_matrices/  CSV + PNG for all_gt and held_out
#   plots/               qualitative GT-vs-prediction figure (CELL 27)
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")
PLOTS_DIR             = os.path.join(RESULTS_ROOT, "plots")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR,
          CONFUSION_MATRIX_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "Prompt_ID", "Prompt_Type",
    "n_gt", "n_prompt_gt", "area_S_px2", "area_M_px2", "area_L_px2", "median_gt_area_px2",
    "n_detections_r1_pre_nms", "n_detections_r1_post_nms",
    "n_self_pos", "self_pos_scores", "n_self_neg", "self_neg_scores", "second_run",
    "n_detections_pre_nms", "image_width", "image_height",
    "npz_file", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)

In [ ]:
# =============================================================================
# CELL 5 - SAM3 MODEL + PROCESSOR
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)


In [ ]:
# =============================================================================
# CELL 6 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================
# Unchanged logic from the original E02_1 notebook (it worked); only wrapped
# into functions and given comments, same as E02_2's CELL 7.
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<n>.txt) and a flat one (labels/<n>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])


In [ ]:
# =============================================================================
# CELL 7 - IoU
# =============================================================================
# Unchanged from the original notebook - it was already correct and vectorised.
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


print("IoU helpers ready.")


In [ ]:
# =============================================================================
# CELL 8 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE BUG IN THE OLD CODE (both E02_1 and E02_2 originally shared this bug)
# The old matcher did:
#     best_gt = argmax(IoU[pred])          <- global best, ignoring availability
#     if best_iou >= thr and best_gt free: match
# So if the globally best GT was already taken, the prediction was thrown away
# even when ANOTHER, still free GT had IoU >= threshold. That under-counted TPs
# (recall too low, false positives too high) on images with clustered plants.
#
# THE CORRECT RULE (implemented here, same as E02_2)
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32)  # IoU of the accepted match for every prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool)  # track which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable")  # confidence, high -> low

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # nothing left to match
        # Consider ONLY currently unmatched GT boxes (this is the fix).
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))  # best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")


In [ ]:
# =============================================================================
# CELL 9 - RESIZE HELPER (whole-image, no tiling)
# =============================================================================
# E02_1 sends the WHOLE image to SAM3 in one pass (no tiles). Very large UAV
# orthomosaics are downscaled to MAX_DIM on the longest side purely for SAM3
# input size / speed; predictions are rescaled back to full resolution
# immediately after inference (see CELL 11). Unchanged from the original
# E02_1 code.
# =============================================================================

def resize_for_sam3(img, max_dim=MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale


print(f"Resize helper ready (MAX_DIM={MAX_DIM}).")


In [ ]:
# =============================================================================
# CELL 10 - SIZE-BASED EXEMPLAR SELECTION (S / M / L)
# =============================================================================
# Size = box AREA (width x height) in full-resolution pixels. ALL GT boxes of the
# image are candidates - boxes cut by the image border are treated like any other
# GT box.
#   S = smallest area
#   L = largest area (another box than S)
#   M = among the remaining boxes, the one whose area is closest to the MEDIAN
#       area of ALL GT boxes of the image
# Ties are broken by the lower GT index -> fully deterministic.
# Fewer than 3 GT boxes: 1 box -> [S], 2 boxes -> [S, L].
# The prompt order sent to SAM3 is always S, M, L.
# =============================================================================

def box_areas(boxes):
    """(N,4) [x1,y1,x2,y2] -> (N,) areas in px^2."""
    b = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def select_size_exemplars(gt_boxes):
    """
    Input : gt_boxes (G,4) full-resolution GT boxes of ONE image
    Output: dict with
              'indices' - GT indices in prompt order (S, M, L)
              'roles'   - ["S", "M", "L"] (shorter when G < 3)
              'areas'   - area of each selected box
              'median_area', 'all_areas'
    """
    areas = box_areas(gt_boxes)
    n = len(areas)
    if n == 0:
        raise ValueError("The image has no GT boxes -> no size-based exemplars.")

    small = int(np.argmin(areas))                          # first index on ties
    if n == 1:
        indices, roles = [small], ["S"]
    else:
        masked = areas.astype(np.float64).copy()
        masked[small] = -np.inf
        large = int(np.argmax(masked))                     # first index on ties, != S
        if n == 2:
            indices, roles = [small, large], ["S", "L"]
        else:
            dist = np.abs(areas.astype(np.float64) - float(np.median(areas)))
            dist[[small, large]] = np.inf
            medium = int(np.argmin(dist))                  # first index on ties
            indices, roles = [small, medium, large], ["S", "M", "L"]

    return {
        "indices": indices,
        "roles": roles,
        "areas": areas[indices],
        "median_area": float(np.median(areas)),
        "all_areas": areas,
    }


def format_size_prompt_id(selection):
    """e.g. {'indices': [12, 5, 40], 'roles': ['S','M','L']} -> 'S12+M5+L40'."""
    return "+".join(f"{r}{i}" for r, i in zip(selection["roles"], selection["indices"]))


# --- quick self-test on a toy image --------------------------------------------
_toy = np.array([[0, 0, 10, 10], [0, 0, 50, 50], [0, 0, 30, 30],
                 [0, 0, 20, 20], [0, 0, 40, 40]], dtype=np.float32)   # areas 100..2500
_sel = select_size_exemplars(_toy)
assert _sel["indices"] == [0, 2, 1], _sel          # S = 10x10, M = 30x30 (median), L = 50x50
print("Size-based exemplar selection ready. Toy example:", format_size_prompt_id(_sel))

In [ ]:
# =============================================================================
# CELL 11 - SAM3 WHOLE-IMAGE INFERENCE (no tiling, positive + negative box prompts)
# =============================================================================
# This is the E02_1 method, KEPT AS IT IS:
#   - resize whole image to MAX_DIM
#   - scale exemplar GT boxes into that resized space
#   - single SAM3 forward pass, no tiling, no exemplar-strip composition
#     (the exemplar plants are already visible in the same image, so they can
#     be passed directly as positive box prompts -- unlike E02_2, which needs
#     the strip trick because a tile may not contain the exemplar plant)
#   - scale predicted boxes back up to full-res
#
# SAM3 is called once per ROUND (round 1: the 3 S/M/L boxes, all positive;
# round 2: those 3 + the positive self-prompts with label 1 and the negative
# self-prompts with label 0) at CONFIDENCE_THRESHOLD (0.30, fixed from the
# start); NMS is applied afterwards, same protocol as E02_2.
# =============================================================================

def _to_numpy(x):
    """Handles tensors (incl. bf16/fp16), lists of tensors, or plain arrays."""
    if torch.is_tensor(x):
        if x.dtype in (torch.bfloat16, torch.float16):
            x = x.float()
        return x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        if len(x) > 0 and torch.is_tensor(x[0]):
            x = [t.float() if t.dtype in (torch.bfloat16, torch.float16) else t for t in x]
            return torch.stack([t.detach().cpu() for t in x]).numpy()
        return np.array(x)
    return np.array(x)


def run_sam3_whole_image(image: Image.Image, pos_boxes_fullres: list,
                         neg_boxes_fullres: list = None,
                         threshold: float = CONFIDENCE_THRESHOLD,
                         mask_threshold: float = MASK_THRESHOLD):
    """
    pos_boxes_fullres: POSITIVE [x1,y1,x2,y2] boxes (ORIGINAL image pixel coords)
    neg_boxes_fullres: NEGATIVE boxes (same coords), or None for none at all

    Returns: pred_boxes_fullres (np.ndarray Nx4), pred_scores (np.ndarray N),
             all boxes/scores with score >= threshold. Masks are requested from
             SAM3 internally (needed by post-processing) but never stored.
    """
    img_w, img_h = image.size
    image_sam, sam_scale = resize_for_sam3(image, MAX_DIM)

    neg_boxes_fullres = list(neg_boxes_fullres) if neg_boxes_fullres is not None else []
    all_boxes_fullres = list(pos_boxes_fullres) + neg_boxes_fullres
    input_boxes_xyxy = [[float(c) * sam_scale for c in box] for box in all_boxes_fullres]
    input_boxes = [input_boxes_xyxy]
    input_boxes_labels = [[1] * len(pos_boxes_fullres)        # 1 = positive prompt
                          + [0] * len(neg_boxes_fullres)]     # 0 = negative prompt

    inputs = sam3_processor(
        images=image_sam,
        input_boxes=input_boxes,
        input_boxes_labels=input_boxes_labels,
        return_tensors="pt",
    ).to(sam3_model.device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    pred_boxes_np = _to_numpy(results["boxes"]).reshape(-1, 4)
    pred_scores_np = _to_numpy(results["scores"]).reshape(-1)

    # Boxes came back in image_sam's (resized) coordinate space -> rescale to full-res
    if len(pred_boxes_np) > 0:
        pred_boxes_np = pred_boxes_np / sam_scale

    # masks are never kept (KEEP_MASKS = False) -> drop the reference immediately
    del results, outputs, inputs

    return pred_boxes_np, pred_scores_np


print("SAM3 whole-image inference helper ready (no tiling, positive/negative box prompts).")


In [ ]:
# =============================================================================
# CELL 12 - NMS
# =============================================================================
# Even without tiling, SAM3 can occasionally propose more than one overlapping
# box for the same plant in a single forward pass. NMS keeps the highest-
# scoring box of each overlapping group. The NMS IoU threshold is fixed
# (NMS_IOU_THRESHOLD = 0.40). It is used TWICE in this notebook: on the round-1
# detections when picking the self-prompts (CELL 13) and on the final detections
# during evaluation. The raw pre-NMS detections of both rounds stay untouched on
# disk -- same protocol as E02_2 (no tile provenance here, there are no tiles).
# =============================================================================

def nms(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep - list of kept indices, highest score first.
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold.
    """
    n = len(boxes)
    if n == 0:
        return []
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep = []
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        order = list(rest[ious <= iou_threshold])
    return keep


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes', 'scores' of the surviving detections SORTED BY
    SCORE (high -> low). Sorting by score means that applying an operating
    confidence threshold later is just a prefix selection.
    """
    keep = np.array(nms(run["boxes"], run["scores"], iou_threshold), dtype=int)
    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "n_pre_nms": int(len(run["scores"])),
    }


print("NMS ready. NMS IoU threshold (fixed):", NMS_IOU_THRESHOLD)


In [ ]:
# =============================================================================
# CELL 13 - SELF-PROMPT SELECTION (round 1 -> round 2)
# =============================================================================
#   round-1 detections
#     -> NMS (NMS_IOU_THRESHOLD) so near-duplicates cannot both be picked
#     -> drop every box that is just a prompt plant again: IoU >=
#        SELF_PROMPT_EXCLUDE_IOU (0.50) with ANY of the S/M/L prompt boxes
#     -> POSITIVE self-prompts: the N_SELF_POSITIVES (2) HIGHEST-confidence boxes.
#        They are picked FIRST and removed from the pool, so a box can never be a
#        positive and a negative at the same time (this matters when every round-1
#        box is below UNRELIABLE_MAX_SCORE).
#     -> NEGATIVE self-prompts: of the remaining boxes, the ones that are not
#        reliable enough to confidently call Rumex (confidence <
#        UNRELIABLE_MAX_SCORE = 0.50), ranked by confidence, LOWEST first, take
#        N_SELF_NEGATIVES (2).
#     -> round 2 gets S/M/L + the positives with label 1 and the negatives with
#        label 0.
# All of these are SAM3's own predictions, NOT ground truth: a false positive fed
# back as a positive example (or a true plant fed back as a negative) can pull
# round 2 off, so the per-image table also reports the round-1 metrics.
# =============================================================================

def select_self_prompts(nms_run, prompt_boxes,
                        n_pos=N_SELF_POSITIVES, n_neg=N_SELF_NEGATIVES,
                        exclude_iou=SELF_PROMPT_EXCLUDE_IOU,
                        unreliable_max_score=UNRELIABLE_MAX_SCORE,
                        confidence_threshold=CONFIDENCE_THRESHOLD):
    """
    Input : nms_run      - output of apply_nms_to_run on the ROUND-1 detections
                           (boxes/scores sorted by score, high -> low)
            prompt_boxes - the S/M/L boxes used in round 1 (full-res coords)
    Output: dict with
              'pos_boxes' (p,4) / 'pos_scores' (p,) - POSITIVE self-prompts
              'neg_boxes' (n,4) / 'neg_scores' (n,) - NEGATIVE self-prompts
              'n_eligible'    how many post-NMS boxes were eligible at all
              'n_on_prompt'   how many were dropped for sitting on a prompt plant
              'n_unreliable'  how many eligible leftovers were below the threshold
    """
    boxes = np.asarray(nms_run["boxes"], dtype=np.float32).reshape(-1, 4)
    scores = np.asarray(nms_run["scores"], dtype=np.float32).reshape(-1)

    keep = scores >= confidence_threshold
    boxes, scores = boxes[keep], scores[keep]

    prompt_boxes = np.asarray(prompt_boxes, dtype=np.float32).reshape(-1, 4)
    if len(boxes) and len(prompt_boxes):
        best_prompt_iou = compute_iou_matrix(boxes, prompt_boxes).max(axis=1)
        on_prompt = best_prompt_iou >= exclude_iou
    else:
        on_prompt = np.zeros(len(boxes), dtype=bool)

    eligible_boxes, eligible_scores = boxes[~on_prompt], scores[~on_prompt]

    # ---- POSITIVE self-prompts: highest confidence, picked first ---------------
    order_desc = np.argsort(-eligible_scores, kind="stable")
    pos_idx = order_desc[:n_pos]
    pos_boxes, pos_scores = eligible_boxes[pos_idx], eligible_scores[pos_idx]

    # ---- NEGATIVE self-prompts: unreliable leftovers, lowest confidence --------
    leftover = np.setdiff1d(np.arange(len(eligible_scores)), pos_idx)   # never a positive
    unreliable = leftover[eligible_scores[leftover] < unreliable_max_score]
    order_asc = unreliable[np.argsort(eligible_scores[unreliable], kind="stable")]
    neg_idx = order_asc[:n_neg]                                         # lowest first
    neg_boxes, neg_scores = eligible_boxes[neg_idx], eligible_scores[neg_idx]

    return {
        "pos_boxes": pos_boxes.reshape(-1, 4), "pos_scores": pos_scores.reshape(-1),
        "neg_boxes": neg_boxes.reshape(-1, 4), "neg_scores": neg_scores.reshape(-1),
        "n_eligible": int(len(eligible_scores)),
        "n_on_prompt": int(on_prompt.sum()),
        "n_unreliable": int(len(unreliable)),
    }


print(f"Self-prompt selection ready: top {N_SELF_POSITIVES} round-1 predictions as POSITIVE, "
      f"the {N_SELF_NEGATIVES} lowest-confidence ones below {UNRELIABLE_MAX_SCORE} as NEGATIVE "
      f"(IoU < {SELF_PROMPT_EXCLUDE_IOU} to the S/M/L prompts).")

In [ ]:
# =============================================================================
# CELL 14 - PRE-NMS DETECTION STORAGE (both rounds)
# =============================================================================
# For every image we store
#   - the ROUND-1 pre-NMS detections,
#   - the POSITIVE and NEGATIVE self-prompt boxes and their confidences,
#   - the FINAL (round-2) pre-NMS detections; when round 2 was skipped because no
#     eligible self-prompt existed, the final arrays are the round-1 ones and
#     second_run is False,
# all in original-image coordinates, at CONFIDENCE_THRESHOLD, before NMS.
# The offline evaluation (CELL 16 onwards) therefore never needs SAM3 again.
# Masks are never stored.
# =============================================================================

def run_npz_path(image_id):
    """Path of the NPZ holding the detections of one image."""
    return os.path.join(RAW_DETECTIONS_DIR, f"{safe_filename(image_id)}.npz")


def save_run_detections(image_id, r1, self_prompts, final, second_run,
                        gt_boxes, selection, image_size):
    """
    Write one image's two-round result to NPZ. The file is self-contained: it also
    stores the GT boxes and the S/M/L prompt indices, so the whole offline
    evaluation and the final plot can run without re-opening label files.
    """
    path = run_npz_path(image_id)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        prompt_indices=np.array(selection["indices"], dtype=np.int32),   # order: S, M, L
        prompt_roles=np.array(selection["roles"]),                       # ["S","M","L"]
        prompt_areas=np.asarray(selection["areas"], dtype=np.float32),
        median_gt_area=np.array(float(selection["median_area"]), dtype=np.float32),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        # ---- round 1 -----------------------------------------------------------
        boxes_round1=np.asarray(r1["boxes"], dtype=np.float32).reshape(-1, 4),
        scores_round1=np.asarray(r1["scores"], dtype=np.float32).reshape(-1),
        # ---- self-prompts taken from round 1 ------------------------------------
        self_pos_boxes=np.asarray(self_prompts["pos_boxes"], dtype=np.float32).reshape(-1, 4),
        self_pos_scores=np.asarray(self_prompts["pos_scores"], dtype=np.float32).reshape(-1),
        self_neg_boxes=np.asarray(self_prompts["neg_boxes"], dtype=np.float32).reshape(-1, 4),
        self_neg_scores=np.asarray(self_prompts["neg_scores"], dtype=np.float32).reshape(-1),
        second_run=np.array(bool(second_run)),
        # ---- final (round 2, or round 1 when round 2 was skipped) ---------------
        boxes=np.asarray(final["boxes"], dtype=np.float32).reshape(-1, 4),
        scores=np.asarray(final["scores"], dtype=np.float32).reshape(-1),
    )
    return path


def load_run_detections(path):
    """Read one image's NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "prompt_indices": z["prompt_indices"].astype(int),
            "prompt_roles": [str(r) for r in z["prompt_roles"]],
            "prompt_areas": z["prompt_areas"].reshape(-1),
            "median_gt_area": float(z["median_gt_area"]),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes_round1": z["boxes_round1"].reshape(-1, 4),
            "scores_round1": z["scores_round1"].reshape(-1),
            "self_pos_boxes": z["self_pos_boxes"].reshape(-1, 4),
            "self_pos_scores": z["self_pos_scores"].reshape(-1),
            "self_neg_boxes": z["self_neg_boxes"].reshape(-1, 4),
            "self_neg_scores": z["self_neg_scores"].reshape(-1),
            "second_run": bool(z["second_run"]),
            "boxes": z["boxes"].reshape(-1, 4),      # FINAL detections
            "scores": z["scores"].reshape(-1),
        }


print("Two-round detection storage ready ->", RAW_DETECTIONS_DIR)

In [ ]:
# =============================================================================
# CELL 15 - MAIN GPU INFERENCE LOOP  (two SAM3 rounds per image)
# =============================================================================
# FOR EACH IMAGE:
#     open the original image ONCE
#     read its GT boxes ONCE
#     pick the S / M / L exemplars by area (CELL 10)
#     ROUND 1 : SAM3 with the 3 GT boxes
#     NMS on the round-1 detections -> pick the positive (highest confidence) and
#         negative (lowest confidence, below UNRELIABLE_MAX_SCORE) self-prompts (CELL 13)
#     ROUND 2 : SAM3 with the 3 GT boxes + the positive self-prompts (label 1) and
#         the negative self-prompts (label 0)
#         (skipped when neither kind exists -> the round-1 output is final)
#     save round 1, the self-prompts and the final detections (NPZ)
#     release the image
#
# Images without any Rumex GT box are skipped. NO metric computation happens here;
# the loop is resumable through runs_manifest.csv.
# =============================================================================

# ---- resume support ---------------------------------------------------------
done_images = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    other_types = sorted(set(_m["Prompt_Type"].astype(str)) - {PROMPT_TYPE})
    if other_types:
        raise RuntimeError(
            f"{RUN_MANIFEST_CSV} already holds results for another prompt setting "
            f"{other_types}. Use a new EXPERIMENT_NAME for {PROMPT_TYPE} "
            "(or delete the old results folder) so the runs are not mixed.")
    done_images = set(_m["image_ID"].astype(str))
    print(f"Resuming: {len(done_images)} images already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
n_without_second_run = 0
image_times = []
n_total_images = len(valid_images)

try:
    for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
        if image_id in done_images:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: already done, skipped.")
            continue
        image_t0 = time.time()

        # ---------------- open the original image exactly once ----------------
        image = Image.open(image_path).convert("RGB")
        img_w, img_h = image.size
        gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
        n_gt = len(gt_boxes)

        if n_gt == 0:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
            image.close(); del image; gc.collect()
            continue

        # ---------------- size-based prompt selection (CELL 10) ----------------
        selection = select_size_exemplars(gt_boxes)
        exemplar_indices = selection["indices"]
        exemplar_boxes_fullres = [gt_boxes[i].tolist() for i in exemplar_indices]
        prompt_id = format_size_prompt_id(selection)

        # ---------------- ROUND 1: the S / M / L GT boxes ----------------------
        r1_boxes, r1_scores = run_sam3_whole_image(
            image, exemplar_boxes_fullres,
            threshold=CONFIDENCE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
        )
        r1 = {"boxes": np.asarray(r1_boxes, dtype=np.float32).reshape(-1, 4),
              "scores": np.asarray(r1_scores, dtype=np.float32).reshape(-1)}

        # ---------------- self-prompts from round 1 (CELL 13) ------------------
        r1_nms = apply_nms_to_run(r1, NMS_IOU_THRESHOLD)
        self_prompts = select_self_prompts(r1_nms, exemplar_boxes_fullres)

        # ---------------- ROUND 2: S/M/L + positive / negative self-prompts ----
        if len(self_prompts["pos_boxes"]) or len(self_prompts["neg_boxes"]):
            round2_pos = exemplar_boxes_fullres + [b.tolist() for b in self_prompts["pos_boxes"]]
            round2_neg = [b.tolist() for b in self_prompts["neg_boxes"]]
            r2_boxes, r2_scores = run_sam3_whole_image(
                image, round2_pos, round2_neg,
                threshold=CONFIDENCE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
            )
            final = {"boxes": np.asarray(r2_boxes, dtype=np.float32).reshape(-1, 4),
                     "scores": np.asarray(r2_scores, dtype=np.float32).reshape(-1)}
            second_run = True
        else:
            # no eligible box -> round 2 would repeat round 1 exactly
            final, second_run = r1, False
            n_without_second_run += 1

        npz_path = save_run_detections(image_id, r1, self_prompts, final, second_run,
                                       gt_boxes, selection, (img_w, img_h))

        areas_by_role = dict(zip(selection["roles"], selection["areas"]))
        run_seconds = time.time() - image_t0
        manifest_writer.writerow({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": image_id,
            "Prompt_ID": prompt_id,
            "Prompt_Type": PROMPT_TYPE,
            "n_gt": n_gt,
            "n_prompt_gt": len(exemplar_indices),
            "area_S_px2": round(float(areas_by_role["S"])),
            "area_M_px2": round(float(areas_by_role["M"])) if "M" in areas_by_role else "",
            "area_L_px2": round(float(areas_by_role["L"])) if "L" in areas_by_role else "",
            "median_gt_area_px2": round(float(selection["median_area"])),
            "n_detections_r1_pre_nms": int(len(r1["scores"])),
            "n_detections_r1_post_nms": int(len(r1_nms["scores"])),
            "n_self_pos": int(len(self_prompts["pos_boxes"])),
            "self_pos_scores": "+".join(f"{s:.3f}" for s in self_prompts["pos_scores"]),
            "n_self_neg": int(len(self_prompts["neg_boxes"])),
            "self_neg_scores": "+".join(f"{s:.3f}" for s in self_prompts["neg_scores"]),
            "second_run": second_run,
            "n_detections_pre_nms": int(len(final["scores"])),
            "image_width": img_w,
            "image_height": img_h,
            "npz_file": os.path.basename(npz_path),
            "inference_seconds": round(run_seconds, 2),
        })
        manifest_file.flush()                  # this image is on disk -> resumable
        n_new_runs += 1

        if len(exemplar_indices) < K_EXEMPLARS:
            print(f"    NOTE: only {n_gt} GT box(es) -> {len(exemplar_indices)} GT prompt(s) "
                  f"({'+'.join(selection['roles'])}).")
        if not second_run:
            print(f"    NOTE: no self-prompt at all ({self_prompts['n_eligible']} eligible, "
                  f"{self_prompts['n_on_prompt']} on a prompt plant, "
                  f"{self_prompts['n_unreliable']} unreliable) -> round 2 skipped, "
                  f"round-1 output kept as final.")

        # ---------------- release the image ------------------------------------
        image.close()
        del image, r1_boxes, r1_scores
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        image_times.append(time.time() - image_t0)
        avg_per_image = float(np.mean(image_times))
        eta = (n_total_images - img_idx) * avg_per_image
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} | {n_gt} GT box(es) | "
              f"prompts={prompt_id} | round1={int(len(r1['scores']))} det "
              f"-> self-prompts: +{int(len(self_prompts['pos_boxes']))} pos "
              f"({', '.join(f'{s:.2f}' for s in self_prompts['pos_scores']) or '-'}) / "
              f"-{int(len(self_prompts['neg_boxes']))} neg "
              f"({', '.join(f'{s:.2f}' for s in self_prompts['neg_scores']) or '-'}) "
              f"-> final={int(len(final['scores']))} det | {run_seconds:.1f}s | "
              f"avg/image={avg_per_image:.1f}s | ETA={eta / 60:.1f} min ({eta / 3600:.2f} h)")
finally:
    manifest_file.close()                      # also closed if the loop crashes

total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new images "
      f"({n_without_second_run} without a second round).")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Detections in: {RAW_DETECTIONS_DIR}")

In [ ]:
# =============================================================================
# CELL 16 - LOAD CACHED PRE-NMS DETECTIONS
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 15 (both rounds), so you can restart the runtime, free
# the GPU and
# still redo the complete evaluation in a few minutes.
# To run the evaluation only: run CELLS 1-4 and 6-14 (skip CELL 5 and CELL 15),
# then continue from here.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[(manifest["experiment_name"] == EXPERIMENT_NAME) &
                    (manifest["Prompt_Type"].astype(str) == PROMPT_TYPE)].copy()
manifest = manifest.drop_duplicates(subset=["image_ID"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_ID"] = str(row["Prompt_ID"])
    run["Prompt_Type"] = str(row["Prompt_Type"])
    runs.append(run)

runs_by_image = {r["image_ID"]: r for r in runs}

print(f"Loaded {len(runs)} runs (= images) for {EXPERIMENT_NAME} ({PROMPT_TYPE}).")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes:", int(sum(len(r['gt_boxes']) for r in runs)))
print("Images with fewer than K GT prompts:",
      int(sum(1 for r in runs if len(r["prompt_indices"]) < K_EXEMPLARS)))
print("Images with a second round:",
      int(sum(1 for r in runs if r["second_run"])), "/", len(runs))
print("Positive self-prompts per image:",
      dict(pd.Series([len(r["self_pos_boxes"]) for r in runs]).value_counts().sort_index()))
print("Negative self-prompts per image:",
      dict(pd.Series([len(r["self_neg_boxes"]) for r in runs]).value_counts().sort_index()))

In [ ]:
# =============================================================================
# CELL 17 - EVALUATION CORE: all_gt AND held_out
# =============================================================================
# all_gt   : classical evaluation, every GT box of the image counts.
#
# held_out : the GT instances that were shown to SAM3 as visual prompts are
#            REMOVED from the GT set, and predictions that fall on those prompt
#            plants are IGNORED (they are neither TP nor FP, they simply do not
#            exist for this evaluation). It answers: "after being shown a few
#            examples, how well does SAM3 find the REMAINING Rumex plants?"
#
# ORDER OF OPERATIONS (important, this is the agreed protocol):
#   1. match predictions to the evaluated (non-prompt) GT with the corrected
#      one-to-one matcher at EVAL_IOU_THRESHOLD = 0.50
#   2. every STILL UNMATCHED prediction whose best IoU with a PROMPT GT box is
#      >= PROMPT_IGNORE_IOU (0.50) becomes IGNORED
#   3. whatever is still unmatched is a false positive
#   Prompt GT boxes themselves are never counted as false negatives.
# =============================================================================

STATUS_FP, STATUS_TP, STATUS_IGNORED = 0, 1, 2


def split_gt_for_mode(gt_boxes, prompt_indices, mode):
    """
    Input : all GT boxes of the image, the indices used as visual prompts, mode
    Output: (evaluated_gt_boxes, prompt_gt_boxes)
            all_gt   -> (all boxes, empty)
            held_out -> (non-prompt boxes, prompt boxes)
    """
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if mode == "all_gt":
        return gt_boxes, np.zeros((0, 4), dtype=np.float32)
    is_prompt = np.zeros(len(gt_boxes), dtype=bool)
    prompt_indices = np.asarray(prompt_indices, dtype=int)
    if len(prompt_indices):
        is_prompt[prompt_indices] = True
    return gt_boxes[~is_prompt], gt_boxes[is_prompt]


def evaluate_run_predictions(pred_boxes, pred_scores, eval_gt_boxes, prompt_gt_boxes,
                             eval_iou=EVAL_IOU_THRESHOLD, ignore_iou=PROMPT_IGNORE_IOU):
    """
    Evaluate ONE prediction set against ONE GT set.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP / STATUS_IGNORED per prediction
      TP, FP, FN, n_ignored, n_eval_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of evaluated GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
      valid_for_macro - False when there is no GT left to evaluate (held_out runs
             in which every plant of the image was used as a prompt)
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    eval_gt_boxes = np.asarray(eval_gt_boxes, dtype=np.float32).reshape(-1, 4)
    prompt_gt_boxes = np.asarray(prompt_gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_eval_gt = len(pred_boxes), len(eval_gt_boxes)
    status = np.full(n_pred, STATUS_FP, dtype=np.int8) # start eveything as FP then valid matches TP then leftover predictions on prompt as ignored and everything still FP

    # step 1 - corrected one-to-one matching against the evaluated GT
    match = match_one_to_one(pred_boxes, pred_scores, eval_gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP # Mark matched predictions as TP

    # step 2 - ignore the leftovers that sit on a PROMPT plant
    # Therefore prompt-ignore logic applies only to predictions that failed to match evaluated GT
    if n_pred and len(prompt_gt_boxes):
        leftover = np.where(status == STATUS_FP)[0] # Find predictions that are still FP (unmatched preds)
        if len(leftover):
            best_prompt_iou = compute_iou_matrix(pred_boxes[leftover], prompt_gt_boxes).max(axis=1)
            status[leftover[best_prompt_iou >= ignore_iou]] = STATUS_IGNORED

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())          # step 3 - the rest are FP
    n_ignored = int((status == STATUS_IGNORED).sum())
    fn = int(n_eval_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_eval_gt if n_eval_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_eval_gt) if n_eval_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "TP": tp, "FP": fp, "FN": fn, "n_ignored": n_ignored,
        "n_eval_gt": n_eval_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
        "valid_for_macro": bool(n_eval_gt > 0),
    }


print("Evaluation core ready (modes:", EVALUATION_MODES, ")")

In [ ]:
# =============================================================================
# CELL 18 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# WHY AP IS COMPUTED DIFFERENTLY FROM PRECISION / RECALL / F1
#
#   AP is an area under the precision-recall curve. That curve is produced by
#   walking through ALL detections ordered by confidence. Truncating the
#   detection list at an operating threshold would simply cut the tail off the
#   curve and report a smaller area - which says nothing about model quality.
#   Therefore AP always uses ALL saved predictions with score >= 0.30
#   (CONFIDENCE_THRESHOLD), after NMS.
#
#   Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point: they
#   answer "if I deploy the detector with confidence >= c, what happens?".
#   Those use only the detections that survive the operating threshold.
#   In this experiment the operating threshold IS the inference threshold (0.30),
#   so both use exactly the same predictions - they still answer different
#   questions (ranking quality vs. deployed behaviour).
#
#   Both numbers can appear in the same row - they answer different questions.
# =============================================================================

# just a helper function that converts NumPy boxes into the format expected by Supervision
def make_detections(boxes, scores=None):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all episodes are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0: # Check whether there is any GT
        return float("nan"), float("nan")
    try:
        # Create the Supervision metric (metric calculator, provides all pred and GTs, calculates the AP results)
        # Supervision exposes map50 for IoU=0.50 and map50_95 for IoUs 0.50:0.95
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead,
        # so it is excluded from means instead of dragging them down.
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


# Its job is not to calculate AP yet, Its job is to prepare: pred and GT for one run so they can later be given to compute_ap(...)
# nms_run: contains your detections after NMS
def ap_inputs_for_run(nms_run, eval_gt, prompt_gt):
    """
    Build the (prediction, GT) episode used for AP of ONE run.
    All post-NMS predictions with score >= CONFIDENCE_THRESHOLD are used;
    in held_out mode the predictions that were IGNORED (they belong to prompt
    plants) are removed first, exactly like in the operating-point evaluation.
    """
    ev = evaluate_run_predictions(nms_run["boxes"], nms_run["scores"], eval_gt, prompt_gt) # marks each pred as TP,FP,IGNORED, it cares only about the ignored predictions for AP calculation bcz in held_out mode , predictions corresponding to prompt rumex must be removed before AP
    keep = ev["status"] != STATUS_IGNORED  # Remove ignored predictions
    # Convert remaining predictions to Supervision format
    return (make_detections(nms_run["boxes"][keep], nms_run["scores"][keep]),
            make_detections(eval_gt))


print("AP helpers ready (AP always uses every prediction >= "
      f"{CONFIDENCE_THRESHOLD} after NMS).")

In [ ]:
# =============================================================================
# CELL 19 - OPERATING-POINT EVALUATION
# =============================================================================
# Choose one confidence threshold, remove predictions below it, then send the
# remaining predictions to CELL 17 for evaluation. This is the function that
# gives Precision, Recall, F1, IoU1, and IoU2 at one operating threshold.
# =============================================================================

def evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, confidence_threshold):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low) => contains the detections after NMS            eval_gt / prompt_gt - from split_gt_for_mode
            confidence_threshold - the operating point being tested
    Output: the dict of evaluate_run_predictions for the thresholded predictions.
    """
    keep = nms_run["scores"] >= confidence_threshold # boolean mask where True entries are selected and the False entries are excluded
    return evaluate_run_predictions(nms_run["boxes"][keep], nms_run["scores"][keep],
                                    eval_gt, prompt_gt)


print("Operating configuration (fixed from the start, no sweep):")
print(f"  Confidence threshold = {CONFIDENCE_THRESHOLD:.2f}")
print(f"  NMS IoU threshold    = {NMS_IOU_THRESHOLD:.2f}")

In [ ]:
# =============================================================================
# CELL 20 - PLOTTING HELPERS
# =============================================================================
# Colours used everywhere:
#   yellow = GT Rumex boxes
#   cyan   = S (smallest) exemplar prompt
#   lime   = M (medium) exemplar prompt
#   orange = L (largest) exemplar prompt
#   magenta= POSITIVE self-prompts (round-1 predictions fed back in round 2)
#   black  = NEGATIVE self-prompts (unreliable round-1 predictions)
#   red    = predictions
# =============================================================================

COLOR_GT, COLOR_PRED = "yellow", "red"
COLOR_SELF_POS, COLOR_SELF_NEG = "magenta", "black"
ROLE_COLORS = {"S": "cyan", "M": "lime", "L": "orange"}
ROLE_NAMES = {"S": "smallest", "M": "medium", "L": "largest"}


def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


def _draw_boxes(ax, boxes, color, linewidth=1.5, linestyle="-", labels=None, fontsize=6):
    """Draw [x1,y1,x2,y2] boxes (display coordinates) on a matplotlib axis."""
    for k, (x1, y1, x2, y2) in enumerate(np.asarray(boxes).reshape(-1, 4)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=color, linewidth=linewidth,
                                       linestyle=linestyle))
        if labels is not None:
            ax.text(x1, y1 - 2, labels[k],
                    color="white" if color in ("red",) else "black",
                    fontsize=fontsize, va="bottom", ha="left",
                    bbox=dict(facecolor=color, edgecolor="none", pad=0.8, alpha=0.85))


def _display_copy(image, max_display_dim=PLOT_MAX_DISPLAY_DIM):
    """Downscaled numpy copy of the image + the factor to map full-res boxes onto it."""
    w, h = image.size
    s = min(1.0, max_display_dim / max(w, h))
    disp_w, disp_h = max(1, int(w * s)), max(1, int(h * s))
    display = np.asarray(image.resize((disp_w, disp_h), Image.BILINEAR))
    to_disp = np.array([disp_w / w, disp_h / h, disp_w / w, disp_h / h], dtype=np.float32)
    return display, to_disp


print("Plotting helpers ready.")

In [ ]:
# =============================================================================
# CELL 21 - PER-IMAGE METRICS  (final = round 2, with round 1 for comparison)
# =============================================================================
# The evaluated predictions are the FINAL ones (round 2, or round 1 when round 2
# was skipped), compared against the original human GT annotations.
# The same metrics are also computed for the ROUND-1 output and written into
# *_round1 columns, so the effect of the self-prompts is visible per image
# (delta_AP50 = AP50 - AP50_round1).
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]


def evaluate_detection_set(det, gt, prompt_indices, mode):
    """AP + operating-point metrics of one detection set (pre-NMS) in one mode."""
    nms_run = apply_nms_to_run(det, NMS_IOU_THRESHOLD)
    eval_gt, prompt_gt = split_gt_for_mode(gt, prompt_indices, mode)
    p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
    ap50, ap5095 = compute_ap([p_det], [g_det])
    ev = evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, CONFIDENCE_THRESHOLD)
    return nms_run, ev, ap50, ap5095


image_rows = []
for run in runs:
    gt = run["gt_boxes"]
    final_det = {"boxes": run["boxes"], "scores": run["scores"]}
    round1_det = {"boxes": run["boxes_round1"], "scores": run["scores_round1"]}

    for mode in EVALUATION_MODES:
        nms_run, ev, ap50, ap5095 = evaluate_detection_set(final_det, gt, run["prompt_indices"], mode)
        _, ev1, ap50_r1, ap5095_r1 = evaluate_detection_set(round1_det, gt, run["prompt_indices"], mode)
        valid = ev["valid_for_macro"]
        nan = float("nan")

        row = {
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": run["image_ID"],
            "Prompt_ID": run["Prompt_ID"],
            "Prompt_Type": run["Prompt_Type"],
            "evaluation_mode": mode,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "nms_iou_threshold": NMS_IOU_THRESHOLD,
            "n_gt": len(gt),
            "n_prompt_gt": int(len(run["prompt_indices"])),
            "n_self_pos": int(len(run["self_pos_boxes"])),
            "self_pos_scores": "+".join(f"{s:.3f}" for s in run["self_pos_scores"]),
            "n_self_neg": int(len(run["self_neg_boxes"])),
            "self_neg_scores": "+".join(f"{s:.3f}" for s in run["self_neg_scores"]),
            "second_run": run["second_run"],
            "median_gt_area_px2": round(run["median_gt_area"]),
            "n_eval_gt": ev["n_eval_gt"],
            "n_predictions_pre_nms": nms_run["n_pre_nms"],
            "n_predictions": ev["n_pred"],
            "n_ignored_predictions": ev["n_ignored"],
            "AP50": ap50 if valid else nan,
            "AP50_95": ap5095 if valid else nan,
            "precision": ev["precision"] if valid else nan,
            "recall": ev["recall"] if valid else nan,
            "F1": ev["F1"] if valid else nan,
            "IoU1": ev["IoU1"] if valid else nan,
            "IoU2": ev["IoU2"] if valid else nan,
            "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
            "valid_for_macro": valid,
            # ---- round 1, same metrics, for comparison --------------------------
            "AP50_round1": ap50_r1 if valid else nan,
            "AP50_95_round1": ap5095_r1 if valid else nan,
            "precision_round1": ev1["precision"] if valid else nan,
            "recall_round1": ev1["recall"] if valid else nan,
            "F1_round1": ev1["F1"] if valid else nan,
            "IoU1_round1": ev1["IoU1"] if valid else nan,
            "IoU2_round1": ev1["IoU2"] if valid else nan,
            "TP_round1": ev1["TP"], "FP_round1": ev1["FP"], "FN_round1": ev1["FN"],
        }
        row["delta_AP50"] = row["AP50"] - row["AP50_round1"] if valid else nan
        row["delta_F1"] = row["F1"] - row["F1_round1"] if valid else nan
        image_rows.append(row)

image_level_df = pd.DataFrame(image_rows).sort_values(
    ["evaluation_mode", "image_ID"]).reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Per-image metrics: {len(image_level_df)} rows "
      f"({image_level_df['image_ID'].nunique()} images x {len(EVALUATION_MODES)} modes) "
      f"-> {IMAGE_LEVEL_CSV}")
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    better = int((sub["delta_AP50"] > 0).sum())
    worse = int((sub["delta_AP50"] < 0).sum())
    same = int((sub["delta_AP50"] == 0).sum())
    print(f"  {mode:9s}: AP50 round1={sub['AP50_round1'].mean():.4f} -> final={sub['AP50'].mean():.4f} "
          f"(mean delta {sub['delta_AP50'].mean():+.4f}) | images better/worse/unchanged: "
          f"{better}/{worse}/{same}")

In [ ]:
# =============================================================================
# CELL 22 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Mean over images and std BETWEEN images of every per-image metric, so every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# it contains.
# =============================================================================

summary_rows = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_exemplars": K_EXEMPLARS,
        "n_self_positives": N_SELF_POSITIVES,
        "n_self_negatives": N_SELF_NEGATIVES,
        "unreliable_max_score": UNRELIABLE_MAX_SCORE,
        "n_images_with_second_round": int(sub["second_run"].sum()),
        "size_measure": SIZE_MEASURE,
        "use_tiling": USE_TILING,
        "max_dim": MAX_DIM,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
        "n_images": int(sub["image_ID"].nunique()),
        "n_images_valid_for_macro": int(sub["valid_for_macro"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = sub[col].mean()
        row[f"{col}_std"] = sub[col].std()      # spread between images
        row[f"{col}_round1_mean"] = sub[f"{col}_round1"].mean()
    row["delta_AP50_mean"] = sub["delta_AP50"].mean()
    row["delta_F1_mean"] = sub["delta_F1"].mean()
    summary_rows.append(row)

experiment_summary_df = pd.DataFrame(summary_rows)
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 23 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# This is NOT the mean of the per-image AP values. All images are handed to
# supervision at once, so every detection of the whole dataset is ranked in ONE
# precision-recall curve. Here one episode = one image.
# =============================================================================

dataset_rows = []
for mode in EVALUATION_MODES:
    pred_list, gt_list, images_used = [], [], set()
    pred_list_r1, gt_list_r1 = [], []
    for run in runs:
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)
        for det, pl, gl in [({"boxes": run["boxes"], "scores": run["scores"]}, pred_list, gt_list),
                            ({"boxes": run["boxes_round1"], "scores": run["scores_round1"]},
                             pred_list_r1, gt_list_r1)]:
            nms_run = apply_nms_to_run(det, NMS_IOU_THRESHOLD)
            p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
            pl.append(p_det)
            gl.append(g_det)
        images_used.add(run["image_ID"])
    ap50, ap5095 = compute_ap(pred_list, gt_list)          # one PR curve over ALL images
    ap50_r1, ap5095_r1 = compute_ap(pred_list_r1, gt_list_r1)
    dataset_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_images": len(images_used),
        "confidence_used_for_AP": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "dataset_AP50": ap50,
        "dataset_AP50_95": ap5095,
        "dataset_AP50_round1": ap50_r1,
        "dataset_AP50_95_round1": ap5095_r1,
    })
    del pred_list, gt_list, pred_list_r1, gt_list_r1
    gc.collect()

dataset_ap_df = pd.DataFrame(dataset_rows)
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print("\nFor comparison, the MEAN of the per-image AP50 values (a different quantity):")
print(image_level_df.groupby("evaluation_mode")["AP50"].mean().to_string())

In [ ]:
# =============================================================================
# CELL 24 - DATASET-LEVEL CONFUSION MATRICES
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
# Counts are pooled over every image at the fixed configuration. In held_out mode
# the S/M/L prompt plants and the detections that were ignored do not appear.
# =============================================================================

confusion_summary = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    tp, fp, fn = int(sub["TP"].sum()), int(sub["FP"].sum()), int(sub["FN"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = safe_f1(precision, recall)

    cm_df = pd.DataFrame(
        [[tp, fn], [fp, np.nan]],
        index=["actual_rumex", "actual_background"],
        columns=["predicted_rumex", "predicted_background"],
    )
    cm_df.to_csv(os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.csv"))

    plot_confusion_matrix(
        tp, fp, fn,
        f"{EXPERIMENT_NAME} - {mode}\nS/M/L prompts | conf={CONFIDENCE_THRESHOLD:.2f}, "
        f"NMS IoU={NMS_IOU_THRESHOLD:.2f}, eval IoU={EVAL_IOU_THRESHOLD:.2f}",
        os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.png"))

    confusion_summary.append({
        "experiment_name": EXPERIMENT_NAME, "evaluation_mode": mode,
        "TP": tp, "FP": fp, "FN": fn,
        "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
        "confidence_threshold": CONFIDENCE_THRESHOLD, "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    })
    print(f"{mode:9s}: TP={tp}  FP={fp}  FN={fn}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

confusion_summary_df = pd.DataFrame(confusion_summary)
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)
print("\nConfusion matrices saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 25 - RECALL PER SIZE GROUP (does the S/M/L mix help across plant sizes?)
# =============================================================================
# For every image the GT boxes are split at that image's MEDIAN area:
#   "area <= median" (the smaller half) and "area > median" (the larger half).
# Matching uses all_gt at the fixed operating point (same matcher as everywhere).
# The prompt plants themselves are reported separately, because SAM3 usually
# re-detects them and they would flatter the recall of their size group.
# =============================================================================

size_rows, prompt_rows = [], []
for run in runs:
    gt = run["gt_boxes"]
    nms_run = apply_nms_to_run({"boxes": run["boxes"], "scores": run["scores"]},
                               NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    match = match_one_to_one(nms_run["boxes"][keep], nms_run["scores"][keep], gt,
                             EVAL_IOU_THRESHOLD)
    found = match["gt_match_pred"] >= 0

    areas = box_areas(gt)
    is_prompt = np.isin(np.arange(len(gt)), run["prompt_indices"])
    small_half = areas <= run["median_gt_area"]
    for name, grp in [("area <= median", small_half), ("area >  median", ~small_half)]:
        mask = grp & ~is_prompt                      # non-prompt GT boxes only
        size_rows.append({
            "image_ID": run["image_ID"], "size_group": name,
            "n_gt_non_prompt": int(mask.sum()), "found": int((found & mask).sum()),
            "recall": float((found & mask).sum() / mask.sum()) if mask.sum() else float("nan"),
        })
    for role, idx in zip(run["prompt_roles"], run["prompt_indices"]):
        prompt_rows.append({"image_ID": run["image_ID"], "role": role,
                            "gt_index": int(idx), "area_px2": float(areas[idx]),
                            "re_detected": bool(found[idx])})

size_group_df = pd.DataFrame(size_rows)
prompt_hits_df = pd.DataFrame(prompt_rows)
size_group_df.to_csv(os.path.join(METRICS_DIR, "size_group_recall.csv"), index=False)
prompt_hits_df.to_csv(os.path.join(METRICS_DIR, "prompt_plants_redetected.csv"), index=False)

pooled = (size_group_df.groupby("size_group")[["n_gt_non_prompt", "found"]].sum()
          .assign(recall_pooled=lambda d: d["found"] / d["n_gt_non_prompt"]))
per_image_mean = size_group_df.groupby("size_group")["recall"].mean().rename("recall_mean_over_images")

print("Recall of NON-PROMPT GT boxes per size group (all_gt matching):")
print(pooled.join(per_image_mean).to_string())
print("\nPrompt plants re-detected (share per role):")
print(prompt_hits_df.groupby("role")["re_detected"].agg(["count", "sum", "mean"]).to_string())
print("\nSaved:", os.path.join(METRICS_DIR, "size_group_recall.csv"), "and",
      os.path.join(METRICS_DIR, "prompt_plants_redetected.csv"))

In [ ]:
# =============================================================================
# CELL 26 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Round 1 prompts          : {K_EXEMPLARS} GT boxes by {SIZE_MEASURE} "
      f"(smallest / medium / largest)")
print(f"Round 2 prompts          : those {K_EXEMPLARS} + up to {N_SELF_POSITIVES} POSITIVE "
      f"self-prompts (highest confidence) + up to {N_SELF_NEGATIVES} NEGATIVE self-prompts "
      f"(lowest confidence below {UNRELIABLE_MAX_SCORE}); IoU < {SELF_PROMPT_EXCLUDE_IOU} to the S/M/L prompts")
print(f"Images with 2 rounds     : {int(sum(1 for r in runs if r['second_run']))} / {len(runs)}")
print(f"Tiling                   : {USE_TILING}  (whole image, longest side -> {MAX_DIM}px)")
print(f"Confidence threshold     : {CONFIDENCE_THRESHOLD:.2f} (fixed; SAM3 run once per image)")
print(f"Mask threshold           : {MASK_THRESHOLD:.2f}")
print(f"NMS IoU threshold        : {NMS_IOU_THRESHOLD:.2f} (fixed)")
print(f"Evaluation IoU           : {EVAL_IOU_THRESHOLD:.2f}")
print(f"Images                   : {len(runs)}")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["evaluation_mode", "AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("ROUND 1 vs FINAL (mean over images)")
print(experiment_summary_df[["evaluation_mode", "AP50_round1_mean", "AP50_mean", "delta_AP50_mean",
                             "F1_round1_mean", "F1_mean", "delta_F1_mean"]].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["evaluation_mode", "dataset_AP50_round1", "dataset_AP50",
                     "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["evaluation_mode", "TP", "FP", "FN",
                            "precision_micro", "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} image NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)

In [ ]:
# =============================================================================
# CELL 27 - QUALITATIVE PLOT: BEST IMAGE, GT (left) vs PREDICTIONS (right)
# =============================================================================
# IMAGE SELECTION (per-image AP50, mode = PLOT_EVALUATION_MODE):
#   1. keep only images that have at least PLOT_MIN_GT_BOXES (7) GT boxes
#      and take the one with the highest AP50
#   2. if NO image has 7 GT boxes: keep the images with the HIGHEST number of GT
#      boxes and take the one among them with the highest AP50
#   ties are broken by F1, then by the number of GT boxes.
#
# Right panel = the FINAL (round-2) post-NMS predictions of that image with
# score >= CONFIDENCE_THRESHOLD, plus its S / M / L prompts (dashed) and the
# positive self-prompts (magenta dashed) and negative self-prompts (black dashed)
# that were fed back into round 2.
# The image is downscaled for DISPLAY only (PLOT_MAX_DISPLAY_DIM).
# =============================================================================

def select_image_for_plot(mode=PLOT_EVALUATION_MODE, min_gt=PLOT_MIN_GT_BOXES):
    """Output: (per-image row, selection_rule_text) or (None, reason)."""
    img_df = image_level_df[(image_level_df["evaluation_mode"] == mode) &
                            image_level_df["AP50"].notna()].copy()
    if img_df.empty:
        return None, "no image with a valid AP50"

    candidates = img_df[img_df["n_gt"] >= min_gt]
    if len(candidates):
        rule = (f"highest AP50 among the {len(candidates)} images "
                f"with >= {min_gt} GT boxes")
    else:
        max_gt = int(img_df["n_gt"].max())
        candidates = img_df[img_df["n_gt"] == max_gt]
        rule = (f"no image has >= {min_gt} GT boxes -> highest AP50 among "
                f"the {len(candidates)} image(s) with the most GT boxes ({max_gt})")

    ranked = candidates.sort_values(["AP50", "F1", "n_gt"], ascending=False, na_position="last")
    print("Top candidates:")
    print(ranked[["image_ID", "n_gt", "Prompt_ID", "n_predictions", "AP50", "F1",
                  "precision", "recall"]].head(5).to_string(index=False))
    return ranked.iloc[0], rule


def plot_best_image(mode=PLOT_EVALUATION_MODE):
    row, rule = select_image_for_plot(mode)
    if row is None:
        print("Nothing to plot:", rule)
        return
    plot_image_id = row["image_ID"]
    run = runs_by_image[plot_image_id]

    nms_run = apply_nms_to_run({"boxes": run["boxes"], "scores": run["scores"]},
                               NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    pred_boxes, pred_scores = nms_run["boxes"][keep], nms_run["scores"][keep]
    gt_boxes_plot = run["gt_boxes"]

    # ---- open the image, downscale for display only ---------------------------
    image_paths = {r[3]: r[1] for r in valid_images}
    if plot_image_id not in image_paths:
        print("Image file not found for", plot_image_id)
        return
    with Image.open(image_paths[plot_image_id]) as im:
        display, to_disp = _display_copy(im.convert("RGB"))

    # ---- figure: GT (left) | predictions (right) -------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(22, 8.5))
    for ax in axes:
        ax.imshow(display)
        ax.axis("off")

    gt_d = gt_boxes_plot * to_disp
    _draw_boxes(axes[0], gt_d, COLOR_GT, linewidth=1.5)
    axes[0].set_title(f"Ground truth: {len(gt_boxes_plot)} Rumex boxes", fontsize=12)

    score_labels = [f"{v:.2f}" for v in pred_scores] if PLOT_SHOW_SCORES else None
    _draw_boxes(axes[1], pred_boxes * to_disp, COLOR_PRED, linewidth=1.5, labels=score_labels)
    for role, idx in zip(run["prompt_roles"], run["prompt_indices"]):
        _draw_boxes(axes[1], gt_d[[idx]], ROLE_COLORS[role], linewidth=2.5, linestyle="--")
    if len(run["self_pos_boxes"]):
        _draw_boxes(axes[1], run["self_pos_boxes"] * to_disp, COLOR_SELF_POS,
                    linewidth=2.5, linestyle="--")
    if len(run["self_neg_boxes"]):
        _draw_boxes(axes[1], run["self_neg_boxes"] * to_disp, COLOR_SELF_NEG,
                    linewidth=2.5, linestyle="--")
    axes[1].set_title(
        f"Final (round {2 if run['second_run'] else 1}): {len(pred_boxes)} boxes | "
        f"prompts {run['Prompt_ID']} + {len(run['self_pos_boxes'])} pos-self "
        f"- {len(run['self_neg_boxes'])} neg-self\n"
        f"AP50={row['AP50']:.3f}  P={row['precision']:.3f}  R={row['recall']:.3f}  "
        f"F1={row['F1']:.3f}  TP={int(row['TP'])} FP={int(row['FP'])} FN={int(row['FN'])}",
        fontsize=12)

    legend_handles = [
        Line2D([0], [0], color=COLOR_GT, lw=2, label="ground truth"),
        Line2D([0], [0], color=COLOR_PRED, lw=2, label="prediction"),
    ] + [Line2D([0], [0], color=ROLE_COLORS[r], lw=2.5, linestyle="--",
                label=f"{r} prompt ({ROLE_NAMES[r]})") for r in run["prompt_roles"]] \
        + ([Line2D([0], [0], color=COLOR_SELF_POS, lw=2.5, linestyle="--",
                   label=f"positive self-prompts ({len(run['self_pos_boxes'])})")]
           if len(run["self_pos_boxes"]) else []) \
        + ([Line2D([0], [0], color=COLOR_SELF_NEG, lw=2.5, linestyle="--",
                   label=f"negative self-prompts ({len(run['self_neg_boxes'])})")]
           if len(run["self_neg_boxes"]) else [])
    fig.legend(handles=legend_handles, loc="lower center", ncol=len(legend_handles),
               fontsize=11, frameon=False)
    fig.suptitle(
        f"{EXPERIMENT_NAME} | {plot_image_id} | mode={mode}\n"
        f"size-based prompts {run['Prompt_ID']} + {len(run['self_pos_boxes'])} pos / "
        f"{len(run['self_neg_boxes'])} neg self-prompts "
        f"| no tiling (MAX_DIM={MAX_DIM}px) | "
        f"conf={CONFIDENCE_THRESHOLD:.2f}, NMS IoU={NMS_IOU_THRESHOLD:.2f}, "
        f"mask thr={MASK_THRESHOLD:.2f}\nselection: {rule}",
        fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.90])

    png_path = os.path.join(PLOTS_DIR, f"best_image_{safe_filename(plot_image_id)}_{mode}.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"\nSelected image : {plot_image_id} ({len(gt_boxes_plot)} GT boxes)")
    print(f"Selection rule : {rule}")
    print(f"Prompts        : {run['Prompt_ID']} + "
          f"{len(run['self_pos_boxes'])} positive self-prompt(s) "
          f"({', '.join(f'{s:.2f}' for s in run['self_pos_scores']) or '-'}) + "
          f"{len(run['self_neg_boxes'])} negative self-prompt(s) "
          f"({', '.join(f'{s:.2f}' for s in run['self_neg_scores']) or '-'})")
    print(f"Figure saved   : {png_path}")


plot_best_image(PLOT_EVALUATION_MODE)